# Lab 01 — AI_COMPLETE Basics

`AI_COMPLETE` is the foundational function for calling LLMs inside Snowflake SQL.
This lab covers the core patterns you'll use in every other lab.

| Item | Detail |
|---|---|
| **Function** | `AI_COMPLETE` |
| **Models** | Claude (`claude-haiku-4-5`, `claude-sonnet-4-5`, `claude-opus-4-5`), Llama (`llama3.1-8b`, `llama3.1-70b`, `llama3.3-70b`, `llama4-maverick`), Mistral (`mistral-large2`, `mistral-medium-3`), OpenAI (`openai-gpt-4.1`, `openai-o4-mini`), Google (`gemini-2.5-flash`), Snowflake (`snowflake-arctic`), DeepSeek (`deepseek-r1`) and more — run `SHOW MODELS IN SNOWFLAKE.MODELS` for the full list |
| **Exam Domain** | 1.0 Gen AI Overview, 2.0 Gen AI & LLM Functions |
| **You'll learn** | Simple prompts, system/user messages, options, prompt patterns, model comparison |

**What You'll Learn**

- Simple string prompts
- System + user message patterns
- AI_COMPLETE options (temperature, max_tokens, top_p)
- Five prompt engineering patterns
- Multi-model comparison

> **Refresh model list:** `CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();` then `SHOW MODELS IN SNOWFLAKE.MODELS;` (requires ACCOUNTADMIN)

## Step 1 — Environment Setup

> **AI SQL Functions** — This lab uses the preferred `AI_` prefixed functions. 
Equivalent legacy functions: `AI_COMPLETE` (replaces `SNOWFLAKE.CORTEX.COMPLETE`).

## Available Models for AI_COMPLETE

`AI_COMPLETE` supports a wide range of LLMs. Run `SHOW MODELS IN SNOWFLAKE.MODELS` to see all models available in your account.

| Provider | Models | Best for |
|---|---|---|
| **Anthropic** | `claude-haiku-4-5` · `claude-sonnet-4-5` · `claude-sonnet-4-6` · `claude-opus-4-5` · `claude-opus-4-6` | General purpose, instruction following, low latency (Haiku) |
| **Meta (Llama)** | `llama3.1-8b` · `llama3.1-70b` · `llama3.1-405b` · `llama3.3-70b` · `llama4-maverick` · `llama4-scout` | Open-weight, cost-effective, wide availability |
| **Mistral** | `mistral-large2` · `mistral-medium-3` · `mistral-7b` · `mixtral-8x7b` | Multilingual, coding, concise outputs |
| **OpenAI** | `openai-gpt-4.1` · `openai-o4-mini` · `openai-gpt-5` | Complex reasoning, structured outputs |
| **Google** | `gemini-2.5-flash` · `gemini-2.5-flash-lite` · `gemini-3-pro` | Multimodal, long context |
| **Snowflake** | `snowflake-arctic` · `snowflake-llama-3.3-70b` | Snowflake-optimised, SQL-focused |
| **DeepSeek** | `deepseek-r1` | Chain-of-thought reasoning |

> **Tip:** Use `claude-haiku-4-5` or `llama3.1-8b` for iterative lab work (fast + low cost). Use `claude-sonnet-4-5` or `llama3.3-70b` for higher quality outputs.

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

-- Database pre-created by SYSADMIN in 00-admin-setup
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Basic Completion

The simplest pattern: model name + prompt string.

In [ ]:
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    'Explain data warehousing in exactly three sentences.'
) AS response;

## Step 3 — System + User Message Pattern

Use the structured message array for role-based prompting. The `system` message sets behavior; the `user` message is the question.

In [ ]:
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    'You are a concise Snowflake expert. Answer in bullet points.

What are the key benefits of Snowflake Cortex AI functions?',
    {'temperature': 0.3, 'max_tokens': 500}
) AS response;

## Step 3b — Multi-Turn Conversation (assistant role)

Multi-turn conversation history is embedded directly in the prompt string. Include system context at the top and prior exchanges as labelled dialogue.

> **Note (legacy):** `SNOWFLAKE.CORTEX.COMPLETE` supported an explicit message-array syntax with `role` objects (`system`/`user`/`assistant`). `AI_COMPLETE` uses a plain string prompt — embed context and history directly in the string.

In [ ]:
-- Multi-turn: embed conversation history directly in the prompt string
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    'You are a Snowflake architecture advisor. Be concise.

User: What is a virtual warehouse?
Assistant: A virtual warehouse is a named cluster of compute resources that executes queries and DML operations.
User: How does it differ from a database?'
) AS multi_turn_response;

## Step 4 — AI_COMPLETE Options Reference

The third argument controls LLM behavior:

| Option | Type | Description |
|---|---|---|
| `temperature` | FLOAT 0.0–1.0 | Controls randomness. 0 = deterministic. |
| `max_tokens` | INT | Maximum output length in tokens. |
| `top_p` | FLOAT 0.0–1.0 | Nucleus sampling threshold. |
| `guardrails` | BOOLEAN | Enable Cortex Guard safety screening. |
| `response_format` | OBJECT | Force output format, e.g. `{type: 'json'}`. |


In [ ]:
SELECT
    'deterministic' AS mode,
    AI_COMPLETE(
        'claude-haiku-4-5',
        'Name 3 Snowflake features.',
        {'temperature': 0.0, 'max_tokens': 100}
    ) AS response
UNION ALL
SELECT
    'creative',
    AI_COMPLETE(
        'claude-haiku-4-5',
        'Name 3 Snowflake features.',
        {'temperature': 0.9, 'max_tokens': 100}
    );

## Step 4b — Additional Options: top_p & Usage Metadata

Beyond `temperature` and `max_tokens`, the options object supports:
- **`top_p`** — nucleus sampling (alternative to temperature); restricts token selection to a cumulative probability
- **Usage metadata** — the response object includes token counts in the `usage` key

In [ ]:
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    'Suggest a creative name for a data pipeline.',
    {'top_p': 0.9, 'max_tokens': 50}
) AS top_p_response;

In [ ]:
-- show_details => true returns a JSON object with choices + usage metadata
SELECT
    resp:choices[0]:messages::STRING AS response,
    resp:usage:prompt_tokens::INT     AS prompt_tokens,
    resp:usage:completion_tokens::INT AS completion_tokens,
    resp:usage:total_tokens::INT      AS total_tokens,
    resp:model::STRING                AS model_used
FROM (
    SELECT AI_COMPLETE(
        model            => 'claude-haiku-4-5',
        prompt           => 'What is Snowpark?',
        model_parameters => {'max_tokens': 100},
        show_details     => true
    ) AS resp
);

## Step 5 — Prompt Engineering Patterns

Five patterns you'll use constantly: zero-shot, few-shot, chain-of-thought, extraction, summarization.

In [ ]:
CREATE OR REPLACE TABLE prompt_experiments (
    experiment_id INT AUTOINCREMENT,
    pattern_name  VARCHAR(100),
    prompt_text   TEXT
);

INSERT INTO prompt_experiments (pattern_name, prompt_text) VALUES
('zero_shot', 'Classify this text as POSITIVE, NEGATIVE, or NEUTRAL: The new feature update broke my workflow completely.'),
('few_shot', 'Classify sentiment. Examples:\nInput: Love it! -> POSITIVE\nInput: Terrible. -> NEGATIVE\nInput: It is okay. -> NEUTRAL\n\nInput: The new feature update broke my workflow completely. ->'),
('chain_of_thought', 'Analyze the sentiment step by step:\n1. Identify key phrases\n2. Determine emotional tone\n3. Give final classification\n\nText: The new feature update broke my workflow completely.'),
('extraction', 'Extract the following from this support ticket in JSON format: {issue_type, severity, product_area}.\nTicket: The dashboard export feature has been timing out for the past 3 days. This is blocking our weekly executive report.'),
('summarization', 'Summarize the key technical decisions in this meeting note in exactly 3 bullet points:\nWe discussed migrating from PostgreSQL to Snowflake. Team agreed on a phased approach starting with analytics workloads. Data engineering will own the pipeline migration while the BI team handles dashboard conversion. Target completion is Q2.');

In [ ]:
SELECT
    pattern_name,
    AI_COMPLETE('claude-haiku-4-5', prompt_text) AS llm_response
FROM prompt_experiments
ORDER BY experiment_id;

## Step 6 — Compare Models

Different models have different strengths. Compare on your specific task.

In [ ]:
SELECT
    'claude-haiku-4-5' AS model,
    AI_COMPLETE('claude-haiku-4-5', 'In one sentence, what makes Snowflake unique?') AS response
UNION ALL
SELECT
    'mistral-large2',
    AI_COMPLETE('mistral-large2', 'In one sentence, what makes Snowflake unique?')
UNION ALL
SELECT
    'llama3.1-70b',
    AI_COMPLETE('llama3.1-70b', 'In one sentence, what makes Snowflake unique?');

## Step 5 — AI_COMPLETE with Images

`AI_COMPLETE` supports three input variants:

| Variant | Syntax | Use Case |
|---|---|---|
| **Text only** | `AI_COMPLETE(model, prompt)` | All text-based tasks (Steps 1–4) |
| **Single image** | `AI_COMPLETE(model, predicate, TO_FILE(stage, path))` | Visual QA, OCR, image analysis |
| **Prompt object** | `AI_COMPLETE(model, PROMPT('...{0}...', file1, ...))` | Multi-image comparison, batch processing |

Image-capable models: `claude-haiku-4-5`, `claude-sonnet-4-5`, `claude-opus-4-5`, `llama4-maverick`, `llama4-scout`, `openai-gpt-4.1`, `gemini-2.5-flash`, `pixtral-large`

> Images must be staged in Snowflake before use. This section uses the `@lab_images` stage (also created in Lab 07).

### Stage Setup

Upload images to `@lab_images` using Snowsight or the PUT command:

```sql
-- PUT from local machine (run in SnowSQL, not Snowsight)
PUT file:///path/to/image.jpg @GENAI_LEARNING.PUBLIC.lab_images;

-- Or upload via Snowsight: Data → Databases → GENAI_LEARNING → PUBLIC → Stages → lab_images → + Files
```

Supported formats: `JPEG`, `PNG`, `WEBP`, `GIF`, `BMP` (max 10 MB)

In [ ]:
-- Create stage (also created in Lab 07 — IF NOT EXISTS is idempotent)
CREATE STAGE IF NOT EXISTS lab_images
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Image files for AI_COMPLETE vision demos';

-- After uploading images, refresh the directory table
ALTER STAGE lab_images REFRESH;

-- Verify staged files
LIST @lab_images;

### 5a — Single Image: Visual Question Answering

Syntax: `AI_COMPLETE(model, predicate, TO_FILE(stage, path))`

The `predicate` is a natural language instruction. The model answers based on the image content.

In [ ]:
-- Replace 'sample.jpg' with a file you have staged in @lab_images
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    'Describe what you see in this image in 2-3 sentences.',
    TO_FILE('@lab_images', 'sample.jpg')
) AS image_description;

In [ ]:
-- Extract structured data from an image (e.g. a chart, receipt, or form)
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    'Extract all visible text and key data from this image. Respond in JSON.',
    TO_FILE('@lab_images', 'sample.jpg')
) AS extracted_data;

### 5b — Prompt Object: Multi-Image with PROMPT()

Syntax: `AI_COMPLETE(model, PROMPT('template {0} ... {1}', file1, file2))`

`{0}`, `{1}` are positional placeholders for `TO_FILE()` references. Mix text columns and image files freely.

In [ ]:
-- Compare two images — replace filenames with your staged images
SELECT AI_COMPLETE(
    'claude-haiku-4-5',
    PROMPT(
        'Compare image {0} and image {1}. What do they have in common and how do they differ?',
        TO_FILE('@lab_images', 'image1.jpg'),
        TO_FILE('@lab_images', 'image2.jpg')
    )
) AS comparison;

In [ ]:
-- Batch classify all images in the stage using the directory table
SELECT
    RELATIVE_PATH                          AS image_file,
    AI_COMPLETE(
        'claude-haiku-4-5',
        PROMPT(
            'Classify this image in 3 words or fewer. Respond in JSON as {"label": "..."}',
            TO_FILE('@lab_images', RELATIVE_PATH)
        )
    ) AS classification
FROM DIRECTORY(@lab_images);

## Key Takeaways

- `AI_COMPLETE` supports both simple string prompts and structured message arrays.
- Use `temperature` (0.0–1.0) to control creativity; lower = more deterministic.
- Five key patterns: zero-shot, few-shot, chain-of-thought, extraction, summarization.
- Apply to table columns with SELECT for batch processing at scale.
- Compare models on your specific task — accuracy, latency, and cost vary.
